In [3]:
# 1. Import Library
import pandas as pd
import numpy as np
import os

# 2. Konfigurasi Path
INSET_PATH = '../../../kamus/inset_final.csv'
OUTPUT_DIR = '../../outputs/SLA'
OUTPUT_FILE = os.path.join(OUTPUT_DIR, 'sla_lexicon_adapted.csv')

os.makedirs(OUTPUT_DIR, exist_ok=True)
np.random.seed(42) # Agar hasil selalu konsisten

# 3. Fungsi Adaptasi Skor (Sesuai Masukan Dosen)
def generate_vader_ratings(mean_val, std_val=1.0):
    """
    Menghasilkan 10 angka bulat yang jika dirata-ratakan hasilnya PAS sesuai mean.
    """
    # Target total haruslah Mean * 10 (misal mean 2.4, total harus 24)
    target_sum = int(round(mean_val * 10))
    
    # Tahap 1: Generate 10 angka awal secara acak (Distribusi Normal)
    ratings = np.random.normal(loc=mean_val, scale=std_val, size=10)
    ratings = np.round(ratings).astype(int)
    ratings = np.clip(ratings, -4, 4)
    
    # Tahap 2: Koreksi Selisih (Agar totalnya PAS)
    current_sum = np.sum(ratings)
    diff = target_sum - current_sum
    
    if diff != 0:
        step = 1 if diff > 0 else -1
        indices = np.arange(10)
        # Penyesuaian angka secara acak sampai selisihnya nol
        while diff != 0:
            np.random.shuffle(indices) # Mengacak urutan index
            for idx in indices:
                new_val = ratings[idx] + step
                # Harus tetap dalam batas [-4, 4]
                if -4 <= new_val <= 4:
                    ratings[idx] = new_val
                    diff -= step
                    if diff == 0: break
                    
    return ratings.tolist()

# 4. Eksekusi Adaptasi
df_inset = pd.read_csv(INSET_PATH)
df_inset['kata'] = df_inset['kata'].astype(str).str.strip().str.lower()

df_sla = pd.DataFrame()
df_sla['kata'] = df_inset['kata']
df_sla['mean'] = (df_inset['skor'] * 0.8).round(1) # Normalisasi dan pembulatan 1 desimal
df_sla['std'] = 1.0
df_sla['raw_ratings'] = df_sla['mean'].apply(lambda m: generate_vader_ratings(m))

# 5. Simpan Hasil
df_sla.to_csv(OUTPUT_FILE, index=False)
print(f"[OUTPUT] Leksikon SLA berhasil diperbaiki dan disimpan: {OUTPUT_FILE}")

[OUTPUT] Leksikon SLA berhasil diperbaiki dan disimpan: ../../outputs/SLA\sla_lexicon_adapted.csv


In [5]:
# Validasi Hasil Perbaikan
df_sla['calculated_mean'] = df_sla['raw_ratings'].apply(lambda x: np.mean(x))
df_sla['diff'] = (df_sla['calculated_mean'] - df_sla['mean']).abs()

print(f"[VALIDASI] Rata-rata selisih (Error): {df_sla['diff'].mean():.10f}")
print(f"[VALIDASI] Selisih Maksimal: {df_sla['diff'].max():.10f}")
print(f"[VALIDASI] Jumlah baris dengan selisih > 0.0001: {(df_sla['diff'] > 0.0001).sum()}")

print("\n[PREVIEW] Mean vs Calculated Mean:")
print(df_sla[['kata', 'mean', 'calculated_mean', 'raw_ratings']].head())

[VALIDASI] Rata-rata selisih (Error): 0.0000000000
[VALIDASI] Selisih Maksimal: 0.0000000000
[VALIDASI] Jumlah baris dengan selisih > 0.0001: 0

[PREVIEW] Mean vs Calculated Mean:
        kata  mean  calculated_mean                       raw_ratings
0        hai   2.4              2.4    [3, 2, 2, 4, 2, 2, 3, 2, 1, 3]
1    merekam   1.6              1.6   [-1, 2, 2, 1, 0, 4, 2, 2, 2, 2]
2  ekstensif   2.4              2.4    [3, 2, 3, 4, 2, 1, 4, 1, 4, 0]
3  paripurna   0.8              0.8  [1, 3, 1, -1, 3, 1, -1, 0, 0, 1]
4     detail   1.6              1.6    [2, 0, 2, 3, 2, 2, 1, 1, 3, 0]
